<a href="https://colab.research.google.com/github/mbahramii/conveyor-stone-detector/blob/main/Stone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile('/content/frames.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("✅ Done!")

In [ ]:
import os
files = os.listdir('/content/frames/')
print(f"تعداد: {len(files)}")
print("نمونه:", sorted(files)[:5])


In [ ]:
import os
print(os.listdir('/content/'))

In [ ]:
import cv2
import numpy as np
import json
import os
from pathlib import Path
from google.colab.patches import cv2_imshow



FRAMES_DIR  = "/content/frames"   # path to frames
OUTPUT_DIR  = "/content/output"   # path to output

# ROI range (from CAM7 camera grid)
ROI_X1, ROI_Y1 = 430, 310
ROI_X2, ROI_Y2 = 848, 490

# Detection parameters
FLOW_WEIGHT        = 0.4    # optical flow weight
DIFF_WEIGHT        = 0.4    # temporal diff weight
TEXTURE_WEIGHT     = 0.2    # texture uniformity weight
HEATMAP_PERCENTILE = 88     # heatmap threshold (higher = stricter)
MIN_AREA           = 800    # minimum stone area (px²)
MIN_AR             = 0.3    # minimum aspect ratio
MAX_AR             = 4.0    # maximum aspect ratio
SHOW_N             = 20     # maximum frames in mosaic



os.makedirs(OUTPUT_DIR, exist_ok=True)

frames_dir  = Path(FRAMES_DIR)
frame_files = sorted(list(frames_dir.glob("*.jpg")) +
                     list(frames_dir.glob("*.png")))
if not frame_files:
    frame_files = sorted(frames_dir.glob("**/*.jpg"))

assert len(frame_files) > 0, f"No frames found in {FRAMES_DIR}!"
print(f"✅ {len(frame_files)} frames loaded")

frames = [cv2.imread(str(f)) for f in frame_files]
grays  = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).astype(np.float32)
          for f in frames]
h, w   = grays[0].shape
n      = len(frames)
print(f"   Resolution: {w}×{h}  |  ROI: x:{ROI_X1}-{ROI_X2}, y:{ROI_Y1}-{ROI_Y2}")




print("⚙ Building background model ...")
bg = np.median(np.array(grays), axis=0).astype(np.float32)
print("   Background ready")




print("⚙ Computing Optical Flow ...")
flows = []
for i in range(n):
    g1   = cv2.GaussianBlur(grays[max(0, i-1)].astype(np.uint8), (5, 5), 1)
    g2   = cv2.GaussianBlur(grays[i].astype(np.uint8), (5, 5), 1)
    flow = cv2.calcOpticalFlowFarneback(g1, g2, None,
                                        0.5, 3, 15, 3, 5, 1.2, 0)
    mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    flows.append(mag)
    if i % 20 == 0:
        print(f"   {i}/{n}")
print("✅ Flow ready")




def detect_stone(idx):
    """
    Returns (annotated_frame, stone_dict or None)

    stone_dict keys:
        center  : [cx, cy]
        bbox    : [x, y, w, h]
        minRect : [width, height, angle_deg]
        area    : float (px²)
    """
    gray = grays[idx]

    # ── Layer 1: Temporal Diff ─────────────────────────────
    diff     = np.abs(gray - bg)
    diff_roi = np.zeros_like(diff)
    diff_roi[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2] = \
        diff[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2]

    # ── Layer 2: Optical Flow (average of ±2 frames) ───────
    lo, hi   = max(0, idx - 2), min(n - 1, idx + 2)
    avg_flow = np.mean(flows[lo:hi + 1], axis=0)
    flow_roi = np.zeros_like(avg_flow)
    flow_roi[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2] = \
        avg_flow[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2]

    # ── Layer 3: Texture Uniformity (low variance = stone) ─
    gray_u8 = gray.astype(np.uint8)
    blur    = cv2.GaussianBlur(gray_u8, (7, 7), 0).astype(np.float32)
    blur2   = cv2.GaussianBlur(
                  (gray_u8.astype(np.float32)) ** 2, (7, 7), 0)
    var_map = np.abs(blur2 - blur ** 2)
    var_roi = np.zeros_like(var_map)
    var_roi[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2] = \
        var_map[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2]

    pos_vals = var_roi[var_roi > 0]
    denom    = (np.percentile(pos_vals, 95) if len(pos_vals) > 0 else 1) + 1e-5
    low_var  = 1.0 - np.clip(var_roi / denom, 0, 1)
    low_var[:ROI_Y1, :]  = 0
    low_var[ROI_Y2:, :]  = 0
    low_var[:, :ROI_X1]  = 0
    low_var[:, ROI_X2:]  = 0

    # ── Combine layers ─────────────────────────────────────
    def norm(x):
        mx = np.percentile(x[x > 0], 98) if x.max() > 0 else 1
        return np.clip(x / (mx + 1e-5), 0, 1)

    combo = (norm(flow_roi) * FLOW_WEIGHT +
             norm(diff_roi) * DIFF_WEIGHT +
             low_var        * TEXTURE_WEIGHT)
    combo = cv2.GaussianBlur(combo.astype(np.float32), (31, 31), 8)

    # ── Threshold & Morphology ─────────────────────────────
    vals = combo[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2]
    thr  = np.percentile(vals, HEATMAP_PERCENTILE)
    mask = (combo > thr).astype(np.uint8) * 255
    mask[:ROI_Y1, :]  = 0
    mask[ROI_Y2:, :]  = 0
    mask[:, :ROI_X1]  = 0
    mask[:, ROI_X2:]  = 0

    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,  5))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (40, 40))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close)

    # ── Contour & Shape Filter ─────────────────────────────
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
    cnts = [c for c in cnts if cv2.contourArea(c) > MIN_AREA]
    if not cnts:
        return frames[idx].copy(), None

    best = None
    for cnt in sorted(cnts, key=cv2.contourArea, reverse=True):
        x, y, bw, bh = cv2.boundingRect(cnt)
        ar = bw / (bh + 1e-5)
        if MIN_AR < ar < MAX_AR:
            best = cnt
            break
    if best is None:
        best = max(cnts, key=cv2.contourArea)

    rect   = cv2.minAreaRect(best)
    box    = np.intp(cv2.boxPoints(rect))
    cx, cy = int(rect[0][0]), int(rect[0][1])
    rw, rh = int(rect[1][0]), int(rect[1][1])
    ang    = float(rect[2])
    x, y, bw, bh = cv2.boundingRect(best)

    stone = dict(
        center  = [cx, cy],
        bbox    = [x, y, bw, bh],
        minRect = [rw, rh, ang],
        area    = float(cv2.contourArea(best))
    )

    # ── Annotation ────────────────────────────────────────
    vis = frames[idx].copy()

    cv2.rectangle(vis, (ROI_X1, ROI_Y1), (ROI_X2, ROI_Y2),
                  (0, 200, 255), 2)
    cv2.putText(vis, "ROI", (ROI_X1 + 5, ROI_Y1 + 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 255), 1)


    cv2.drawContours(vis, [box], 0, (0, 255, 120), 2)
    cv2.circle(vis, (cx, cy), 7, (0, 255, 120), -1)
    cv2.circle(vis, (cx, cy), 7, (255, 255, 255), 1)
    cv2.line(vis, (cx - 14, cy), (cx + 14, cy), (0, 255, 120), 1)
    cv2.line(vis, (cx, cy - 14), (cx, cy + 14), (0, 255, 120), 1)

    cv2.putText(vis, f"{rw}x{rh}px  {ang:.0f}deg",
                (cx - 40, cy - 14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 120), 2)

    cv2.rectangle(vis, (0, 0), (w, 28), (10, 10, 10), -1)
    cv2.putText(vis,
        f"Frame {idx:03d}/{n-1}  |  STONE  |  "
        f"center:({cx},{cy})  size:{rw}x{rh}px  "
        f"angle:{ang:.1f}deg  area:{int(stone['area'])}px2",
        (5, 19), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (180, 230, 200), 1)

    return vis, stone




print("\n🔍 Detecting stones ...")
results    = []
key_thumbs = []

for idx in range(n):
    vis, stone = detect_stone(idx)
    results.append({"frame": frame_files[idx].name, "stone": stone})

    if stone and len(key_thumbs) < SHOW_N:
        thumb = cv2.resize(vis, (424, 263))
        cv2.putText(thumb, f"f{idx}", (5, 255),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        key_thumbs.append(thumb)

    if idx % 10 == 0:
        status = "STONE ✅" if stone else "---"
        print(f"   {frame_files[idx].name}: {status}")

detected = [r for r in results if r["stone"]]
print(f"\n✅ Detection complete: {len(detected)}/{n} frames with stone")



cols = 4
while len(key_thumbs) % cols:
    key_thumbs.append(np.zeros((263, 424, 3), np.uint8))

rows   = [np.hstack(key_thumbs[i:i + cols])
          for i in range(0, len(key_thumbs), cols)]
mosaic = np.vstack(rows)

hdr = np.full((46, mosaic.shape[1], 3), (12, 12, 12), np.uint8)
cv2.putText(hdr,
    f"STONE DETECTION | Flow x Diff x Texture | "
    f"Detected: {len(detected)}/{n} frames | GREEN=minRect CYAN=ROI",
    (8, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.44, (0, 200, 255), 1)

mosaic_final = np.vstack([hdr, mosaic])
mosaic_path  = f"{OUTPUT_DIR}/mosaic.jpg"
cv2.imwrite(mosaic_path, mosaic_final)

print("\n📊 Results mosaic:")
cv2_imshow(mosaic_final)




if detected:
    ws    = [r["stone"]["minRect"][0] for r in detected]
    hs    = [r["stone"]["minRect"][1] for r in detected]
    angs  = [r["stone"]["minRect"][2] for r in detected]
    areas = [r["stone"]["area"]       for r in detected]

    print(f"\n{'─'*50}")
    print(f"  Total frames        : {n}")
    print(f"  Frames with stone   : {len(detected)}")
    print(f"  Accuracy            : {len(detected)/n*100:.0f}%")
    print(f"  Width (minRect) avg={np.mean(ws):.0f}  "
          f"range={min(ws):.0f}–{max(ws):.0f} px")
    print(f"  Height          avg={np.mean(hs):.0f}  "
          f"range={min(hs):.0f}–{max(hs):.0f} px")
    print(f"  Angle           avg={np.mean(angs):.1f}  "
          f"range={min(angs):.1f}–{max(angs):.1f} deg")
    print(f"  Area            avg={np.mean(areas):.0f}  "
          f"range={int(min(areas))}–{int(max(areas))} px2")
    print(f"{'─'*50}")

log_path = f"{OUTPUT_DIR}/stone_log.json"
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n💾 Mosaic: {mosaic_path}")
print(f"💾 JSON log: {log_path}")

In [ ]:
# from google.colab.patches import cv2_imshow
# import cv2
# import numpy as np
# import glob

# # لود فریم‌ها
# all_paths = sorted(glob.glob("/content/frames/frame_*.jpg"))

# # 6 تا نمونه تصادفی
# import random
# samples = random.sample(all_paths, 6)

# for path in samples:
#     img = cv2.imread(path)
#     # ROI
#     cv2.rectangle(img, (450,320), (848,490), (0,200,255), 2)
#     print(path.split('/')[-1])
#     cv2_imshow(img)
import os
print(os.listdir('/content/frames/'))
# import glob
# all_paths = sorted(glob.glob("/content/frames/frame_*.jpg"))
# print(f"تعداد: {len(all_paths)}")
# print(all_paths[:3])